In [ ]:
import math
import random
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn as dglnn
from dgl.dataloading import MultiLayerNeighborSampler, NodeDataLoader

# ============================================================
# 0) GNN：GraphSAGE (mean) 计算节点 embedding
# ============================================================

class GraphSAGE(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, out_dim: int = 64, num_layers: int = 2):
        super().__init__()
        assert num_layers == 2, "这里按论文的 k-hop(=2) 示例写成2层，你也可以扩展。"
        self.conv1 = dglnn.SAGEConv(in_dim, hidden_dim, "mean")
        self.conv2 = dglnn.SAGEConv(hidden_dim, out_dim, "mean")

    def forward(self, blocks: List[dgl.DGLGraph], x: torch.Tensor) -> torch.Tensor:
        h = self.conv1(blocks[0], x)
        h = F.relu(h)
        h = self.conv2(blocks[1], h)
        return h

@torch.no_grad()
def infer_embeddings_minibatch(
    g: dgl.DGLGraph,
    feat: torch.Tensor,
    model: nn.Module,
    fanouts: List[int] = [15, 10],   # 每层采多少邻居（子图太大就采样缩小）
    batch_size: int = 4096,
    device: str = "cpu",
) -> torch.Tensor:
    """
    对所有节点算 embedding（minibatch + 邻居采样）。
    输出 embedding 会 softplus 变成非负，方便 order-embedding 的逐维 <=。
    """
    model.eval()
    g = g.to(device)
    feat = feat.to(device)

    sampler = MultiLayerNeighborSampler(fanouts)
    loader = NodeDataLoader(
        g, torch.arange(g.num_nodes(), device=device),
        sampler, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0
    )

    out_dim = model.conv2.out_feats
    emb = torch.empty(g.num_nodes(), out_dim, device=device)

    for input_nodes, output_nodes, blocks in loader:
        x = feat[input_nodes]
        h = model(blocks, x)
        h = F.softplus(h)  # 非负化 -> order embedding 的逐维比较更自然
        emb[output_nodes] = h

    return emb


# ============================================================
# 1) Subgraph extraction：抽 3-10 节点的中心子图
#    （论文说k-hop邻域，子图大就采样；这里用 2-hop + fanout 采样）
# ============================================================

def sample_center_subgraph_3_10(
    g: dgl.DGLGraph,
    center: int,
    num_hops: int = 2,
    fanouts: List[int] = [8, 6],
    min_nodes: int = 3,
    max_nodes: int = 10,
    max_steps: int = 200,
) -> dgl.DGLGraph:
    """
    目标：每个样本子图节点数严格在 [3,10]。
    做法：
      - 先用层级邻居采样拿一个“中心附近”的节点池
      - 再从这个池里随机扩展/裁剪到 3-10
      - 最后取 induced subgraph
    """
    device = g.device
    seed = torch.tensor([center], device=device)

    # 1) 采样得到候选节点池（k-hop, 每层fanout）
    sampler = dgl.dataloading.MultiLayerNeighborSampler(fanouts[:num_hops])
    input_nodes, output_nodes, blocks = sampler.sample_blocks(g, seed)
    pool = input_nodes  # 包含 center + 采样到的邻居
    pool_list = pool.tolist()

    # 2) 从池里构造 3-10 节点集合（确保包含 center）
    target_n = random.randint(min_nodes, max_nodes)

    selected = {center}
    frontier = [center]
    pool_set = set(pool_list)

    steps = 0
    while len(selected) < target_n and frontier and steps < max_steps:
        steps += 1
        v = frontier.pop(0)
        nbrs = g.successors(v)
        if nbrs.numel() == 0:
            continue
        # 优先选在 pool 里的邻居（更“k-hop”）
        cand = [int(u) for u in nbrs.tolist() if int(u) in pool_set and int(u) not in selected]
        if not cand:
            cand = [int(u) for u in nbrs.tolist() if int(u) not in selected]
        if not cand:
            continue
        u = random.choice(cand)
        selected.add(u)
        frontier.append(u)

    # 不够就从 pool 里补齐
    if len(selected) < target_n:
        rest = [u for u in pool_list if u not in selected]
        random.shuffle(rest)
        for u in rest:
            selected.add(int(u))
            if len(selected) >= target_n:
                break

    # 还是不够（极少见）就随机补
    while len(selected) < min_nodes:
        selected.add(random.randrange(g.num_nodes()))

    nodes = torch.tensor(list(selected), device=device)
    sg = dgl.node_subgraph(g, nodes)  # induced
    sg.ndata["is_center"] = (sg.ndata[dgl.NID] == center).float().unsqueeze(-1)
    return sg


# ============================================================
# 2) PickPatterns：按你截图的 Threshold/Cover/乘积排序/删覆盖/贪心
# ============================================================

def compute_thresholds(emb: torch.Tensor, sigma: int, chi: float) -> torch.Tensor:
    """
    Threshold[i] = 第 i 维 top(chi*sigma) 个值中的最小值
    """
    n, d = emb.shape
    topm = max(1, int(math.floor(chi * sigma)))
    topm = min(topm, n)
    thr = []
    for i in range(d):
        vals = emb[:, i]
        top_vals, _ = torch.topk(vals, k=topm, largest=True)
        thr.append(top_vals.min())
    return torch.stack(thr, dim=0)  # [d]

def remove_dominated_points_topM(emb: torch.Tensor, topM: int = 20000) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    删掉被覆盖(dominated)的 embedding：
      ea 被 eb 覆盖 <=> ea <= eb（逐维）
    这是 O(n^2) 问题，真实大图要近似：
      - 先取“体积（乘积）最大”的 topM 再做 dominance 过滤
    返回 (过滤后的emb, keep_mask(对应原始节点))
    """
    n, d = emb.shape
    scores = torch.sum(torch.log(emb + 1e-12), dim=1)  # log(product)
    M = min(topM, n)
    top_idx = torch.topk(scores, k=M).indices
    sub = emb[top_idx]

    keep_sub = torch.ones(M, dtype=torch.bool, device=emb.device)
    for i in range(M):
        if not keep_sub[i]:
            continue
        ei = sub[i]
        for j in range(M):
            if i == j or not keep_sub[j]:
                continue
            if torch.all(ei <= sub[j]):
                keep_sub[i] = False
                break

    keep_mask = torch.zeros(n, dtype=torch.bool, device=emb.device)
    keep_mask[top_idx[keep_sub]] = True
    return emb[keep_mask], keep_mask

def pickpatterns_select_nodes(
    emb: torch.Tensor,
    k: int,
    sigma: int,
    chi: float = 0.5,
    dominance_topM: int = 20000,
) -> torch.Tensor:
    """
    完整实现你截图的 PickPatterns：
      (1) 删被覆盖的；按乘积(体积)降序
      (2) 维护 Cover[i]，只要某维 a_j > Cover[j] 就选；并更新 Cover = max(Cover, a)
      (再加一个 Threshold 过滤小噪声：只把 a_j >= Threshold[j] 的提升当作有效)
    返回：被选的中心节点 id（原图节点）
    """
    device = emb.device
    n, d = emb.shape

    emb2, keep_mask = remove_dominated_points_topM(emb, topM=dominance_topM)
    kept_idx = torch.nonzero(keep_mask, as_tuple=False).squeeze(1)

    thr = compute_thresholds(emb2, sigma=sigma, chi=chi)

    # 线性序：按 product(a_i) 排序；用 sum(log) 避免溢出
    scores = torch.sum(torch.log(emb2 + 1e-12), dim=1)
    order = torch.argsort(scores, descending=True)
    emb_sorted = emb2[order]
    idx_sorted = kept_idx[order]

    cover = torch.zeros(d, device=device)
    selected = []

    for ei, idx in zip(emb_sorted, idx_sorted):
        if len(selected) >= k:
            break
        improve = (ei > cover) & (ei >= thr)
        if bool(torch.any(improve).item()):
            selected.append(idx)
            cover = torch.maximum(cover, ei)

    if len(selected) == 0:
        return torch.empty(0, dtype=torch.long, device=device)
    return torch.stack(selected)


# ============================================================
# 3) 组装：生成训练集（k个代表中心 -> 每个中心一个3-10节点子图）
# ============================================================

def serialize_subgraph_for_diffusion(
    sg: dgl.DGLGraph,
    feat_key: str = "feat",
) -> Dict[str, torch.Tensor]:
    """
    给 diffusion 用的最简单格式：
      - nids: 原图节点id
      - edges: 子图边列表（局部节点编号）
      - x: 子图节点特征（可选）
      - is_center: 标记中心点
    """
    src, dst = sg.edges()
    data = {
        "nids": sg.ndata[dgl.NID].cpu(),
        "edges": torch.stack([src.cpu(), dst.cpu()], dim=0),  # [2, E]
        "is_center": sg.ndata["is_center"].cpu(),
    }
    if feat_key in sg.ndata:
        data["x"] = sg.ndata[feat_key].cpu()
    return data

def generate_training_dataset(
    g: dgl.DGLGraph,
    feat_key: str = "feat",
    # embedding / gnn
    emb_dim: int = 64,
    hidden_dim: int = 128,
    emb_fanouts: List[int] = [15, 10],
    # subgraph extraction
    sub_fanouts: List[int] = [8, 6],
    min_nodes: int = 3,
    max_nodes: int = 10,
    # PickPatterns
    k_select: int = 5000,
    sigma: int = 20,
    chi: float = 0.5,
    dominance_topM: int = 20000,
    device: str = "cpu",
    save_path: str = "dataset.pt",
) -> Tuple[List[Dict[str, torch.Tensor]], torch.Tensor]:
    """
    输出：
      dataset: List[样本字典] （每个样本就是一个3-10节点中心子图）
      selected_nodes: 被选的中心节点
    """
    assert feat_key in g.ndata, f"g.ndata['{feat_key}'] 不存在"

    # 1) 计算节点 embedding（GNN + 采样）
    feat = g.ndata[feat_key]
    model = GraphSAGE(in_dim=feat.shape[1], hidden_dim=hidden_dim, out_dim=emb_dim).to(device)

    # 你如果已有训练好的模型权重，在这里 load_state_dict 即可
    emb = infer_embeddings_minibatch(
        g, feat, model, fanouts=emb_fanouts, batch_size=4096, device=device
    )

    # 2) PickPatterns 选中心节点
    selected_nodes = pickpatterns_select_nodes(
        emb, k=k_select, sigma=sigma, chi=chi, dominance_topM=dominance_topM
    )

    # 3) 从这些中心节点生成 3-10 节点子图样本
    g_dev = g.to(device)
    dataset = []
    for v in selected_nodes.tolist():
        sg = sample_center_subgraph_3_10(
            g_dev, v, num_hops=2, fanouts=sub_fanouts,
            min_nodes=min_nodes, max_nodes=max_nodes
        )
        # 把原图特征拷到子图（node_subgraph会带NID，但不一定带feat）
        sg.ndata[feat_key] = g_dev.ndata[feat_key][sg.ndata[dgl.NID]]
        dataset.append(serialize_subgraph_for_diffusion(sg, feat_key=feat_key))

    torch.save({"dataset": dataset, "selected_nodes": selected_nodes.cpu()}, save_path)
    return dataset, selected_nodes
